# [센서 실습 27] 도플러 레이더 센서 – RCWL-0516 / HB100

마이크로파 도플러 레이더 **RCWL-0516**의 디지털 출력으로 움직임을 감지하고, **HB100**의 중간주파수(IF) 신호를 FFT로 분석해 **물체의 속도**까지 계산합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
**도플러 효과**: 다가오는 구급차 소리는 높게, 멀어지면 낮게 들립니다. 전파도 마찬가지입니다.
- 레이더가 전파를 쏘면 움직이는 물체에 반사된 전파의 주파수가 조금 바뀜
- 보낸 전파와 받은 전파를 섞으면 그 **차이 주파수(도플러 주파수)** 만 남음

$$f_d = \frac{2 v f_0}{c} \quad\Rightarrow\quad v = \frac{f_d \, c}{2 f_0}$$

- HB100 ($f_0$ = 10.525GHz): **1 m/s당 약 70Hz** (1 km/h당 약 19.5Hz)
- 전파는 얇은 벽·플라스틱을 통과 → **벽 너머·커버 안쪽**에서도 감지 (자동문, 숨은 조명 스위치)
- **RCWL-0516**(약 3.2GHz): 움직임이 있으면 출력이 약 2초 동안 HIGH가 되는 **완성형 모듈**
- **HB100**: 도플러 신호(수 mV)를 그대로 내보냄 → 증폭 후 ADC로 받아 FFT로 주파수를 구하면 **속도** 측정 가능 (스피드건 원리)

### 부품 사양
| 항목 | 내용 |
|---|---|
| RCWL-0516 | 3.2GHz, 감지 거리 5~7m, 디지털 출력 (약 2초 유지), 4~28V |
| HB100 | 10.525GHz, 감지 거리 약 20m, 아날로그 IF 출력 (증폭 필요), 5V |
| 특징 | 빛·온도의 영향을 받지 않음, 얇은 벽 투과 |

## 2. 라즈베리파이 연결 방법
**준비물**: RCWL-0516 모듈 (속도 측정은 HB100 + 연산증폭기 + ADC 필요)

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| RCWL-0516 VIN | 5V (2번 핀) |  |
| RCWL-0516 GND | GND (6번 핀) |  |
| RCWL-0516 OUT | GPIO17 (11번 핀) | 출력은 3.3V라 바로 연결 |

모듈 뒷면(부품이 없는 면)이 감지 방향입니다. 근처에 금속이 있으면 감지 범위가 달라집니다.

**인터페이스 활성화**
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
도플러 주파수 ↔ 속도 변환 함수입니다. 속도 측정 실습(7단계)에서 사용합니다.

In [ ]:
C = 3e8               # 빛의 속도 (m/s)
F0 = 10.525e9         # HB100 송신 주파수 (Hz)

def doppler_to_speed(fd_hz):
    return fd_hz * C / (2 * F0)                       # m/s

def speed_to_doppler(v_ms):
    return 2 * v_ms * F0 / C

print(f"1 m/s → {speed_to_doppler(1):.1f} Hz,  100 Hz → {doppler_to_speed(100):.2f} m/s ({doppler_to_speed(100) * 3.6:.1f} km/h)")

### 4-2. 실제 센서 클래스
RCWL-0516 출력은 디지털이므로 `DigitalInputDevice`로 읽습니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealRCWL0516:
    def __init__(self, pin=17):
        from gpiozero import DigitalInputDevice
        self.dev = DigitalInputDevice(pin, pull_up=None, active_state=True)

    def read(self):
        return {"motion": int(self.dev.value)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 시간 | 가정한 상황 |
|---|---|
| 15 ~ 22초 | 벽 너머 복도를 사람이 지나감 (벽 투과 감지) |
| 50초 | 선풍기 바람에 커튼이 잠깐 흔들림 (1초) |
| 80 ~ 95초 | 사람이 방에 들어와 서성임 |
| 출력 유지 | 마지막 움직임 후 2초 동안 HIGH | RCWL-0516 기본 동작 |

속도 측정(7단계)은 HB100 IF 신호를 **1.4 m/s로 걷는 사람**, **15 km/h 자전거** 두 경우로 만들어 FFT로 분석합니다.

In [ ]:
class VirtualRCWL0516:
    HOLD = 2.0

    def __init__(self, dt=1.0, seed=27):
        self.dt, self.t, self.last = dt, 0.0, -99.0
        self.moves = [(15, 22), (50, 51), (80, 95)]

    def read(self):
        if any(a <= self.t < b for a, b in self.moves):
            self.last = self.t
        out = int(self.t - self.last < self.HOLD)
        self.t += self.dt
        return {"motion": out}

def virtual_hb100_if(v_ms, fs=2000, sec=1.0, seed=0):
    """HB100 중간주파수(IF) 신호를 증폭·ADC 한 것처럼 만들기"""
    rng = np.random.default_rng(seed)
    t = np.arange(int(fs * sec)) / fs
    return 0.5 * np.sin(2 * np.pi * speed_to_doppler(v_ms) * t) + rng.normal(0, 0.2, len(t))

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualRCWL0516(dt=1.0)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
1초 간격으로 2분 동안 움직임을 기록합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 120               # 측정 횟수
INTERVAL = 1.0               # 측정 간격(초)

sensor = RealRCWL0516() if USE_REAL_SENSOR else VirtualRCWL0516(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
1. 움직임 감지 횟수를 셉니다. (1초짜리 짧은 감지는 '약한 움직임'으로 따로 표시)
2. **HB100 속도 측정**: IF 신호를 FFT 하여 가장 강한 주파수 → 속도로 변환합니다.

In [ ]:
starts = df[df.motion.diff() == 1].time_s.tolist()
ends = df[df.motion.diff() == -1].time_s.tolist()
for s, e in zip(starts, ends):
    kind = "약한 움직임(커튼 등)" if e - s <= 3 else "사람 움직임"
    print(f"{s:5.0f} ~ {e:5.0f}초  ({e - s:.0f}초)  {kind}")

print("\n[HB100 속도 측정]")
FS_IF = 2000
results = []
for name, v in [("걷는 사람", 1.4), ("자전거", 15 / 3.6)]:
    sig = virtual_hb100_if(v, fs=FS_IF, seed=1)
    amp = np.abs(np.fft.rfft(sig * np.hanning(len(sig))))
    freq = np.fft.rfftfreq(len(sig), 1 / FS_IF)
    fd = freq[np.argmax(amp[5:]) + 5]                  # 아주 낮은 주파수(직류 근처)는 제외
    results.append((name, freq, amp, fd))
    print(f"{name}: 도플러 {fd:.0f} Hz → 속도 {doppler_to_speed(fd):.2f} m/s ({doppler_to_speed(fd) * 3.6:.1f} km/h)  [실제 {v * 3.6:.1f} km/h]")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].step(df.time_s, df.motion, "k-", where="post"); ax[0].fill_between(df.time_s, df.motion, step="post", color="lightgray")
ax[0].set_yticks([0, 1]); ax[0].set_title("RCWL-0516 움직임 감지"); ax[0].set_xlabel("시간 (초)"); ax[0].grid(alpha=0.3)
for (name, freq, amp, fd), st in zip(results, ["k-", "k--"]):
    ax[1].plot(freq, amp / amp.max(), st, lw=1, label=f"{name} ({fd:.0f} Hz)")
ax[1].set_xlim(0, 400); ax[1].set_xlabel("도플러 주파수 (Hz)"); ax[1].set_ylabel("상대 크기")
ax[1].set_title("HB100 IF 신호의 주파수 분석 → 속도"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "도플러센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "RCWL-0516 / HB100",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/radar", payload)
# client.disconnect()

## 직접 해 보기
- 시속 60km 자동차의 도플러 주파수는 몇 Hz인가요? 그것을 측정하려면 ADC 샘플링을 최소 몇 Hz로 해야 할까요?
- PIR 센서(8번)와 레이더 센서의 장단점을 표로 비교해 보세요.
- 1초짜리 감지를 무시하는 필터를 넣어 사람 움직임만 남겨 보세요.